# Prefilling River Basin Districts and Competent Authorities data

This notebook builds a prefilled version of the data to be submitted for the 4th River Basin Management Plans, under the *River Basin Districts and Competent Authorities* dataflow.

All the SQL lives in the precompiled DuckDB catalog `wise_rbdsuca.duckdb`, committed in this repository:

| schema | content |
| --- | --- |
| `prefill` | views resolving the DiscoData / data lake sources for one country |
| `reference` | views over the reference datasets used by the reference QCs |
| `qc` | the quality checks, used by the *check* notebook |
| `meta` | the `stable_record_id` macro, the dataset inventory and the export scripts |

The result is a SQLite file (descriptive data) and an OGC GeoPackage (spatial data) conforming to the data model of the 4th reporting cycle, documented in the [WFD reporting documentation](https://eeadata.github.io/WISE.WFD.Documentation/TestingPhase/WFDRiverBasinDistrictsAndCompetentAuthorities.html).

> `wise_rbdsuca.duckdb` ships precompiled - no build step needed. If you edit anything under `sql/`, run `python build_catalog.py` from the repository root to refresh it.

In [1]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


## 1. Initial setup

Opens the catalog and loads the DuckDB extensions the views depend on (`spatial`, `httpfs`, `azure`, `sqlite`).

In [2]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "wise_local_qc.py").exists())
sys.path.insert(0, str(ROOT))

import geopandas as gpd
import ipywidgets as widgets
from ipyleaflet import GeoData, LayersControl, Map, basemaps

import wise_local_qc as wq

# avoid a locked-file error if this cell is re-run without restarting the kernel
if "con" in globals():
    try:
        con.close()
    except Exception:
        pass

con = wq.connect(ROOT / "wise_rbdsuca.duckdb")
wq.current_parameters(con)

{'country_code': 'AT', 'cycle_year': '2022', 'reference_cycle_year': '2022'}

## 2. Available tables

Everything the catalog can resolve. Nothing is downloaded yet: these are views.

In [3]:
wq.list_datasets(con)

,schema_name,table_name,kind,description
0,prefill,CompetentAuthority,descriptive,Prefilled descriptive data for the selected co...
1,prefill,RiverBasinDistrictCompetentAuthority,descriptive,Prefilled descriptive data for the selected co...
2,reference,Country,reference,Reference dataset from the reference reporting...
3,reference,RiverBasinDistrictWFD,reference,Reference dataset from the reference reporting...
4,prefill,SPATIAL_RiverBasinDistrict,spatial,Prefilled spatial data for the selected countr...


## 3. Country selection

Pick the country and the reporting cycle to prefill, then run the next cell to apply the selection. The selection is pushed to the catalog as DuckDB session variables, which every view reads through `getvariable()`.

In [4]:
countries_selection = widgets.Dropdown(options=wq.COUNTRIES, value="AT", description="Country:")
cycle_selection = widgets.Dropdown(options=["2022", "2016", "2010"], value="2022", description="Cycle:")
widgets.VBox([countries_selection, cycle_selection])

In [5]:
wq.set_parameters(con, country_code=countries_selection.value, cycle_year=cycle_selection.value)
wq.current_parameters(con) 

{'country_code': 'AT', 'cycle_year': '2022', 'reference_cycle_year': '2022'}

## 4. Query the data available for the selected country

Each cell below triggers the download of only what it needs.

In [6]:
wq.preview(con, "prefill.CompetentAuthority")

,euCACode,competentAuthorityName,competentAuthorityNameNL,competentAuthorityNameNLLanguage,acronym,street,city,country,postcode,url,record_id
0,ATCA4,"Federal Ministry for Climate Action, Environme...","Bundesministerium für Klimaschutz, Umwelt, Ene...",ger,BMK,Radetzkystraße 2,Vienna,Austria,1030,http://www.bmk.gv.at/,cd83590e-e8e7-b66c-bf2c-fbbc7358c87e
1,ATCA1,"Federal Ministry of Agriculture, Regions and T...","Bundesministerium für Landwirtschaft, Regionen...",ger,BMLRT,Stubenring 1,Vienna,Austria,1010,http://www.bmlrt.gv.at/,019e5052-9f49-3655-1937-964de7565ee0
2,ATCA11_1,Landeshauptmann (Governor) von Burgenland,Landeshauptmann von Burgenland,ger,kein Akronym,Europaplatz 1,Eisenstadt,Austria,7000,https://www.burgenland.at/,12bd5f84-788c-8e7c-13f8-d9cd8bdf15c8
3,ATCA12_1,Landeshauptfrau (Governor) von Niederösterreich,Landeshauptfrau von Niederösterreich,ger,kein Akronym,Landhausplatz 1,St. Pölten,Austria,3109,http://www.noel.gv.at/,5a975f41-f3c0-6682-3176-946b1d320e41
4,ATCA13_1,Landeshauptmann (Governor) von Wien,Landeshauptmann von Wien,ger,kein Akronym,Lichtenfelsgasse 2,Vienna,Austria,1010,http://www.wien.gv.at/,702ac535-64f7-6ce5-3382-aed4b2d0c878
5,ATCA2,"Federal Ministry of Social Affairs, Health, Ca...","Bundesministerium für Soziales, Gesundheit, Pf...",ger,kein Akronym,Stubenring 1,Vienna,Austria,1010,https://www.sozialministerium.at/,efe7e65a-a5e8-08a2-a2d5-51fd428f3ce0
6,ATCA21_1,Landeshauptmann (Governor) von Kärnten,Landeshauptmann von Kärnten,ger,kein Akronym,Arnulfplatz 1,Klagenfurt,Austria,9021,http://www.ktn.gv.at/,e709e861-7c86-1faf-2b74-e0d41c25cb41
7,ATCA22_1,Landeshauptmann (Governor) der Steiermark,Landeshauptmann der Steiermark,ger,kein Akronym,Hofgasse 16,Graz,Austria,8010,http://www.steiermark.at/,26f5a76b-bca9-3061-b94a-64afeda453ba
8,ATCA3,Federal Ministry for Digital and Economic Affairs,Bundesministerium für Digitalisierung und Wirt...,ger,BMDW,Stubenring 1,Vienna,Austria,1010,http://www.bmdw.gv.at/,041b9ea6-5e88-ae48-4e30-5dade8b6bbbf
9,ATCA31_1,Landeshauptmann (Governor) von Oberösterreich,Landeshauptmann von Oberösterreich,ger,kein Akronym,Landhausplatz 1,Linz,Austria,4021,http://www.land-oberoesterreich.gv.at,b27eeb7b-f191-8bd5-ea2e-657e8018a0c0


In [7]:
wq.preview(con, "prefill.RiverBasinDistrictCompetentAuthority")

,euRBDCode,euCACode,mainRole,record_id
0,AT5000,ATCA1,monitoringOfGroundwater,596eff34-9d0b-378b-3187-ac1e49e2394b
1,AT5000,ATCA1,enforcementOfRegulations,1f207ff9-6fa0-0a62-5434-af78d8b9bf2e
2,AT2000,ATCA1,monitoringOfSurfaceWater,821de7ce-346e-2801-556b-e77e2efea4dc
3,AT2000,ATCA1,publicParticipation,fe554d0c-cec4-6ecf-ed19-2ac16f1a1ddf
4,AT1000,ATCA1,monitoringOfSurfaceWater,32c2b0b6-890c-cb72-b33e-0a0caef4447b
5,AT1000,ATCA1,publicParticipation,16ebc2ce-ffe4-a5e0-8967-e1cf402b7b50
6,AT2000,ATCA1,economicAnalysis,f4a545cf-5324-6ca2-6405-e6f64a99c5ae
7,AT1000,ATCA1,economicAnalysis,1a463a45-b0e8-88cc-c511-ca0522d4be03
8,AT5000,ATCA1,pressureAndImpactAnalysis,867cb320-4fa5-cdad-050d-4f1c8953690c
9,AT5000,ATCA1,assessmentOfStatusOfSurfaceWater,91913e38-c709-1d69-33d8-f6371f0a41e3


In [8]:
con.sql("SELECT * EXCLUDE (geometry_polygon) FROM prefill.SPATIAL_RiverBasinDistrict").df()

,inspireIdLocalId,inspireIdNamespace,inspireIdVersionId,thematicIdIdentifier,thematicIdIdentifierScheme,beginLifespanVersion,endLifespanVersion,predecessorsIdentifier,predecessorsIdentifierScheme,successorsIdentifier,...,nameLanguage,designationPeriodBegin,designationPeriodEnd,zoneType,specialisedZoneType,legalBasisName,legalBasisLink,legalBasisLevel,link,record_id
0,1000,AT.WFD.2022.RiverBasinDistrict,None,AT1000,euRBDCode,None,None,None,None,None,...,ger,2010-03-30,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,8748e8ea-8480-fc7f-55c6-62288f6c656b
1,2000,AT.WFD.2022.RiverBasinDistrict,None,AT2000,euRBDCode,None,None,None,None,None,...,ger,2010-03-30,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,e4d461a3-7a2c-d0e8-f67c-7b2419d3f336
2,5000,AT.WFD.2022.RiverBasinDistrict,None,AT5000,euRBDCode,None,None,None,None,None,...,ger,2010-03-30,None,riverBasinDistrict,internationalRiverBasinDistrict,None,None,None,None,8225fd5a-8674-aa37-f89c-a10698634399


## 5. Export to SQLite and GeoPackage

`record_id` is a deterministic UUID derived from the business key of each row (`meta.stable_record_id`). It is not imported into Reportnet 3, but the QCs report it, so it has to stay identical between the export and the QC run.

Export statements are `COPY` / `CREATE TABLE` statements and therefore cannot be stored as views; they are persisted as text in `meta.scripts` and executed from there.

In [9]:
output_dir = ROOT / "output" / countries_selection.value
sqlite_path = output_dir / "RiverBasinDistrictsAndCompetentAuthorities.sqlite"
geopackage_path = output_dir / "RiverBasinDistrict.gpkg"

wq.export_prefill(con, sqlite_path, geopackage_path)

Did you mean "SPATIAL_RiverBasinDistrict"?

LINE 17:     FROM prefill.SPATIAL_ProtectedArea
                  ^


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?
Did you mean "CompetentAuthority"?


['SPATIAL_RiverBasinDistrict',
 'CompetentAuthority',
 'RiverBasinDistrictCompetentAuthority']

In [10]:
con.sql("SELECT name, target, sql_text FROM meta.scripts WHERE kind = 'export'").df()

,name,target,sql_text
0,CompetentAuthority,sqlite,-- export: CompetentAuthority -> SQLite\n-- Re...
1,derogation,sqlite,-- export: derogation -> SQLite\n-- Requires a...
2,exceedance,sqlite,-- export: exceedance -> SQLite\n-- Requires a...
3,monitoringresult,sqlite,-- export: monitoringresult -> SQLite\n-- Requ...
4,parameter,sqlite,-- export: parameter -> SQLite\n-- Requires an...
5,qualityandmonitoring,sqlite,-- export: qualityandmonitoring -> SQLite\n-- ...
6,RiverBasinDistrictCompetentAuthority,sqlite,-- export: RiverBasinDistrictCompetentAuthorit...
7,SPATIAL_ProtectedArea,geopackage,-- export: SPATIAL_ProtectedArea -> OGC GeoPac...
8,SPATIAL_RiverBasinDistrict,geopackage,-- export: SPATIAL_RiverBasinDistrict -> OGC G...


## 6. Review the exported spatial data

In [11]:
rbd_gdf = gpd.read_file(geopackage_path, layer="RiverBasinDistrict").to_crs(epsg=4326)

m = Map(
    center=(54.5260, 15.2551),
    zoom=4,
    basemap=basemaps.OpenStreetMap.Mapnik,
    scroll_wheel_zoom=True,
    layout={"height": "600px"},
)
m.add(LayersControl())
m.add(
    GeoData(
        geo_dataframe=rbd_gdf,
        name="RiverBasinDistrict",
        style={"color": "blue", "fillColor": "blue", "opacity": 0.5, "weight": 1.9, "dashArray": "5, 5"},
        hover_style={"fillColor": "red", "fillOpacity": 0.5},
    )
)
m

Map(center=[54.526, 15.2551], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [12]:
con.close()